In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import pandas as pd
import numpy as np
import os

dt = pd.read_csv(os.path.join(path,'Q1_data.csv'))

In [ ]:
dt.head()

In [ ]:
dt.info()

In [ ]:
dt.describe()

In [ ]:
from matplotlib import pyplot as plt

# Year distribution
plt.figure(figsize=(10, 5))
plt.hist(dt['Delivery_Time'].dropna(), bins=30, edgecolor='black', color='orange')
plt.title('Delivery_Time Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
dt = dt.drop('Order_ID',axis=1)

In [ ]:
dt = dt.dropna(subset=['Delivery_Time','Traffic_Level','Time_of_Day','Weather'])
dt['Courier_Experience_yrs'] = dt['Courier_Experience_yrs'].fillna( int(dt['Courier_Experience_yrs'].mean()) )
dt.isna().sum()


In [ ]:
dt = dt.drop_duplicates()
print(dt.duplicated().sum())

dt

In [ ]:
from sklearn.preprocessing import LabelEncoder

lb = LabelEncoder()

dt['Weather'] = lb.fit_transform( dt['Weather'] )
dt['Traffic_Level'] = lb.fit_transform( dt['Traffic_Level'] )
dt['Time_of_Day'] = lb.fit_transform( dt['Time_of_Day'] )
dt['Vehicle_Type'] = lb.fit_transform( dt['Vehicle_Type'] )

dt

In [ ]:
from sklearn.preprocessing import StandardScaler

X,y = dt.drop('Delivery_Time',axis=1), dt['Delivery_Time']

std = StandardScaler()
X = std.fit_transform(X)


In [ ]:
# There are no imbalance in the data
# The data is well distributed and a linear model was trained on it (not categorical).

In [ ]:
from sklearn.model_selection import train_test_split

X_train,X_test,y_train,y_test = train_test_split(X,y)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor()

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Train and predict
    model.fit(X_fold_train, y_fold_train)
    y_fold_pred = model.predict(X_fold_val)

    # Calculate metrics
    mae_scores.append(mean_absolute_error(y_fold_val, y_fold_pred))

mae_scores = np.array(mae_scores)

print(f"5-Fold CV Results:")
print(f"MAE: {mae_scores.mean():,.2f} Minutes")

In [ ]:
# Feature importance
feature_importance = pd.DataFrame({
    'feature': ['Distance_km', 'Weather', 'Traffic_Level', 'Time_of_Day',
       'Vehicle_Type', 'Preparation_Time_min', 'Courier_Experience_yrs'],
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
plt.scatter(model.predict(X_test),y_test, color='red')
plt.plot([10,120],[10,120],color='green')
plt.xlabel('Acual Y')
plt.ylabel('Predicted Y')
plt.show()

In [ ]:
from IPython.display import clear_output
%pip install catboost -q
clear_output()

from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import KFold
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor

import numpy as np

rf_model = RandomForestRegressor(random_state=42)

cb_model = CatBoostRegressor(random_state=42, verbose=0)

kfold = KFold(n_splits=5, shuffle=True, random_state=42)

mae_scores_averaged = []

for train_idx, val_idx in kfold.split(X_train):
    X_fold_train, X_fold_val = X_train[train_idx], X_train[val_idx]
    y_fold_train, y_fold_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    rf_model.fit(X_fold_train, y_fold_train)
    cb_model.fit(X_fold_train, y_fold_train)

    rf_preds = rf_model.predict(X_fold_val)
    cb_preds = cb_model.predict(X_fold_val)

    averaged_predictions = (rf_preds + cb_preds) / 2

    fold_mae = mean_absolute_error(y_fold_val, averaged_predictions)
    mae_scores_averaged.append(fold_mae)

print(f"[+] Averaged Models MAE: {np.mean(mae_scores_averaged)}")